<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        GBFS: el Detective que Sigue el Olfato 🧸🎯
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Edición Didáctica: Para Dummies (No Expertos)
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 04 • Algoritmos de Búsqueda (Dummies)
      </span><br>
      <span style="color: #92400e; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

---
## ¿Qué vamos a aprender aquí? 🎈

En el cuaderno anterior, DFS y BFS buscaban **a ciegas**: no les importaba si el archivo se llamaba `informe_final.pdf` o `nomina.xlsx`. Ahora vamos a construir un buscador con **olfato de detective**: usará el propio nombre del archivo como pista para decidir qué carpeta abrir primero.

A este buscador con olfato se le llama **Greedy Best-First Search (GBFS)**, o "Búsqueda Voraz Primero el Mejor". La palabra *voraz* es clave: significa que **siempre elige lo que le parece más prometedor EN ESE MOMENTO**, sin pensar si esa decisión le convendrá más adelante.

---
## 1. Construyendo el "Olfato": ¿Qué tan Parecidos son Dos Nombres? 👃

Para darle olfato a nuestro detective, usamos una herramienta de Python que compara qué tan parecidos son dos textos letra por letra: `difflib.SequenceMatcher`. Cuanto más parecidos, más "cerca" cree el detective que está del objetivo.

In [ ]:
from difflib import SequenceMatcher

def que_tan_parecido(nombre_carpeta, nombre_objetivo):
    """Devuelve un número entre 0 (nada parecido) y 1 (idéntico)."""
    return SequenceMatcher(None, nombre_carpeta.lower(), nombre_objetivo.lower()).ratio()

objetivo = "informe_final.pdf"

for nombre in ["informe_final.pdf", "Informes", "informe_preliminar.pdf", "ProyectosIngenieria", "nomina.xlsx"]:
    parecido = round(que_tan_parecido(nombre, objetivo), 2)
    print(f"'{nombre}' se parece un {parecido*100:.0f}% a '{objetivo}'")

Nuestro detective preferirá siempre abrir primero la carpeta con **mayor porcentaje de parecido** (más "olfato").

In [ ]:
# La misma oficina del cuaderno 01
oficina = {
    "Gerencia": {
        "Actas": {"acta_enero.docx": None, "acta_febrero.docx": None},
        "Estrategia": {"plan_2026.pptx": None, "presupuesto.xlsx": None},
    },
    "Finanzas": {
        "Contabilidad": {"balance.xlsx": None, "Auditorias": {"auditoria_2024.pdf": None}},
        "Tesoreria": {"flujo_caja.xlsx": None},
    },
    "ProyectosIngenieria": {
        "ProyectoNorte": {
            "planos.dwg": None,
            "Informes": {"informe_avance.pdf": None, "informe_final.pdf": None},
        },
        "ProyectoSur": {"Informes": {"informe_preliminar.pdf": None}},
    },
    "RecursosHumanos": {"nomina.xlsx": None, "contrato_empleado.pdf": None},
}
print("Oficina lista, con", len(oficina), "armarios principales.")

---
## 2. Programando al Detective Voraz (GBFS) 🕵️

En vez de una pila o una fila normales, el detective usa una **libreta ordenada por olfato**: siempre saca primero la carpeta que le "huele" mejor (más parecida al nombre que busca), sin importar cuántas puertas ya abrió para llegar hasta ahí.

In [ ]:
import heapq

def buscar_gbfs(mapa, nombre_objetivo):
    contador = 0  # para desempatar sin comparar diccionarios entre sí
    libreta = []
    for nombre, contenido in mapa.items():
        distancia_olfato = 1 - que_tan_parecido(nombre, nombre_objetivo)
        heapq.heappush(libreta, (distancia_olfato, contador, [nombre], nombre, contenido))
        contador += 1

    orden_visitado = []
    while libreta:
        distancia_olfato, _, camino, nombre_actual, contenido_actual = heapq.heappop(libreta)
        orden_visitado.append(nombre_actual)

        if nombre_actual == nombre_objetivo:
            return camino, orden_visitado

        if contenido_actual is not None:
            for nombre_hijo, contenido_hijo in contenido_actual.items():
                d = 1 - que_tan_parecido(nombre_hijo, nombre_objetivo)
                contador += 1
                heapq.heappush(libreta, (d, contador, camino + [nombre_hijo], nombre_hijo, contenido_hijo))

    return None, orden_visitado


camino_gbfs, orden_gbfs = buscar_gbfs(oficina, objetivo)

print("🕵️ Resultado de GBFS:")
print("   Camino encontrado:", " → ".join(camino_gbfs))
print(f"   Carpetas/archivos revisados: {len(orden_gbfs)}")
print("   Orden en que se revisaron:", orden_gbfs)

### 🤔 ¿Nuestro detective fue tan listo como parecía?

Mira con atención el **orden en que se revisaron** las carpetas. Es muy probable que el detective se haya desviado primero hacia `Finanzas → Contabilidad → Auditorias`, atraído por el "olfato" de que esos nombres comparten letras con `informe_final.pdf` (¡aunque no tengan absolutamente nada que ver!). Sólo después de equivocarse, terminó llegando a la carpeta correcta.

**Esa es la gran debilidad de GBFS:** su "olfato" puede engañarlo. A veces será súper rápido (si el olfato es bueno), y otras veces perderá tiempo en pistas falsas — y **nunca hay garantía** de cuál de las dos cosas va a pasar.

---
## 3. Comparando a los Tres Buscadores 📊

In [ ]:
from collections import deque

def buscar_dfs(mapa, nombre_objetivo):
    pila = [([nombre], nombre, contenido) for nombre, contenido in mapa.items()]
    orden = []
    while pila:
        camino, nombre_actual, contenido_actual = pila.pop()
        orden.append(nombre_actual)
        if nombre_actual == nombre_objetivo:
            return camino, orden
        if contenido_actual is not None:
            for h, c in contenido_actual.items():
                pila.append((camino + [h], h, c))
    return None, orden

def buscar_bfs(mapa, nombre_objetivo):
    cola = deque([([nombre], nombre, contenido) for nombre, contenido in mapa.items()])
    orden = []
    while cola:
        camino, nombre_actual, contenido_actual = cola.popleft()
        orden.append(nombre_actual)
        if nombre_actual == nombre_objetivo:
            return camino, orden
        if contenido_actual is not None:
            for h, c in contenido_actual.items():
                cola.append((camino + [h], h, c))
    return None, orden

_, orden_dfs = buscar_dfs(oficina, objetivo)
_, orden_bfs = buscar_bfs(oficina, objetivo)

print(f"{'Buscador':<10} | {'Carpetas/archivos revisados':<28} | ¿Usa el nombre como pista?")
print("-" * 65)
print(f"{'DFS':<10} | {len(orden_dfs):<28} | No")
print(f"{'BFS':<10} | {len(orden_bfs):<28} | No")
print(f"{'GBFS':<10} | {len(orden_gbfs):<28} | Sí (parecido de nombres)")

---
## 🎯 Resumen Relámpago ⚡

| Idea Clave | Explicación en 5 segundos |
|---|---|
| **GBFS** | Un buscador "con olfato": siempre abre primero la carpeta cuyo nombre se parece más al que busca. |
| **"Voraz"** | Decide sólo por lo que parece mejor EN ESE MOMENTO, sin pensar en el camino ya recorrido. |
| **Ventaja** | Cuando el olfato es bueno, puede ser mucho más rápido que buscar a ciegas. |
| **Debilidad** | El olfato puede engañarse con nombres parecidos que no tienen nada que ver, haciéndolo perder tiempo en pistas falsas — y **nunca garantiza** encontrar el camino más corto. |

**Siguiente paso:** en el cuaderno **03, "A\*: el GPS que Nunca se Equivoca"**, le daremos a nuestro detective algo más que olfato: también recordará cuánto camino ya recorrió, para tomar SIEMPRE la mejor decisión posible.

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Algoritmos de Búsqueda (Edición Didáctica Para Dummies)</i>
  </p>
</div>